In [1]:
# 1. wconcept 크롤링 준비

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import time

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

driver.quit()

In [2]:
# 2. wconcept > 베스트 카테고리 접속

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import time

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수:", len(items))

driver.quit()

찾은 상품 수: 200


In [10]:
# 3. wconcept > 베스트 카테고리 접속 > 상세정보 수집

# 판매가 기준 상위 20개 상품 조회
# 할인율이 30% 이상인 상품 조회 & 내림차순 정렬
# 특정 브랜드 (임의) 상품만 조회, 판매가 기준 오름차순 정렬
# 리뷰수가 100이상인 상품의 평균 판매가 조회
# 브랜드별 상품수 & 평균할인률 구하고 상품수가 많은 브랜드 10개 조회

# 소비자가 / 판매가 / 할인률 / 브랜드명 / 상품명 / 리뷰갯수 / 좋아요갯수

# 네이밍정보 1개 -> (브랜드명 + 상품명)
# 가격정보 1개 -> (소비자가 + 할인률 + 판매가)
# 서브정보 1개 -> (평점 + 리뷰갯수 + 좋아요갯수)

# 테이블을 나누는 기준 : 식별성을 가지고 있어야 하는 대상이 어떤 것인가?
# sakila : 렌탈 / 카테고리 / 영화

# 브랜드 전용 테이블
# 상품관련 테이블

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수:", len(items))

def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

for idx, item in enumerate(items, 1) :
    # 브랜드 & 상품정보
    brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
    brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()

    
    # 소비자가 & 판매가 & 할인률정보
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    try : 
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
    except :
        original_price = 0

    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
    except :
        sale_price = 0

    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
    except :
        discount_rate = 0

    # 평점 & 리뷰수 & 좋아요
    stats = item.find_elements(By.CSS_SELECTOR, "span.stats")

    if stats :
        stats = stats[0]

        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text.strip())
        except :
            rating = 0
    
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text.strip())
        except :
            review_count = 0
    
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like").text.strip())
        except :
            like_count = 0
    else :
        rating = 0.0
        review_count = 0
        like_count = 0

    # 상품링크
    img_src = item.find_element(By.CSS_SELECTOR, "button.area-img img").get_attribute("src")
    
    match = re.search(r"/(\d{9})_", img_src)

    if match :
        product_no = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_no}"
    else :
        product_no = None
        product_url = ""

    #ca
        
    # print(img_src)
    # print(idx, brand_name, product_name, original_price, sale_price, discount_rate, rating, review_count, like_count, product_url)

driver.quit()

찾은 상품 수: 200


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"div.swiper-slide"}
  (Session info: chrome=145.0.7632.160); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xcb7dd3
	0xcb7e14
	0xac1db0
	0xb0c20a
	0xb0c4ab
	0xb019d1
	0xb2ea84
	0xb018c4
	0xb2ec04
	0xb4b621
	0xb2e7d6
	0xb00049
	0xb00e04
	0xf16924
	0xf11bf7
	0xf2f5a0
	0xcd0f58
	0xcd891d
	0xcc0648
	0xcc0812
	0xcaa21a
	0x75e0fcc9
	0x77b282ae
	0x77b2827e


In [9]:
# 4. wconcept > 베스트 카테고리 접속 > 상세정보 수집 > MySQL 데이터 전송

# 테이블 2개
# 브랜드 전용 테이블
# 상품 전용 테이블
# 파이썬 -> MySQL 데이터 전송 (pymysql)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time
import pymysql

# mysql 접속정보
conn = pymysql.connect(
    host="127.0.0.1",
    user="root",
    password="Hayeon123!",
    db="wconcept_crawling02",
    charset="utf8mb4"
)

cursor = conn.cursor()

# 가상 브라우저 세팅정보
service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수:", len(items))

def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

for idx, item in enumerate(items, 1) :
    # 브랜드 & 상품정보
    brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
    brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()

    # 소비자가 & 판매가 & 할인률정보
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    try : 
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
    except :
        original_price = 0

    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
    except :
        sale_price = 0

    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
    except :
        discount_rate = 0

    # 평점 & 리뷰수 & 좋아요
    stats = item.find_elements(By.CSS_SELECTOR, "span.stats")

    if stats :
        stats = stats[0]

        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text.strip())
        except :
            rating = 0
    
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text.strip())
        except :
            review_count = 0
    
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like").text.strip())
        except :
            like_count = 0
    else :
        rating = 0.0
        review_count = 0
        like_count = 0

    # 상품링크
    img_src = item.find_element(By.CSS_SELECTOR, "button.area-img img").get_attribute("src")
    
    match = re.search(r"/(\d{9})_", img_src)

    if match :
        product_no = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_no}"
    else :
        product_no = None
        product_url = ""

    # mysql 테이블 저장 - brands
    cursor.execute("""
        INSERT INTO brands (brand_name) VALUES (%s) ON DUPLICATE KEY UPDATE brand_name = VALUES(brand_name)
    """, (brand_name))

    cursor.execute("SELECT brand_id FROM brands WHERE brand_name = %s", (brand_name,))
    brand_id = cursor.fetchone()[0]

    #mysql 테이블 저장 -categories
    cursor.execute("""
        INSERT INTO categories (brand_id, category_name, category_id) VALUES (%s, %s, %s) 
    """, (brand_id, category_name, category_id))
    
    # mysql 테이블 저장 - products
    cursor.execute("""
        INSERT INTO products (brand_id, rank_no, product_name, original_price,
        sale_price, discount_rate, rating, review_count, like_count, product_url
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (brand_id, idx, product_name, original_price, sale_price, discount_rate, rating, review_count,
         like_count, product_url))

    print(f"{idx}위 저장 완료: {brand_name} /{product_name} / {product_no}")

conn.commit()
conn.close()

driver.quit()

찾은 상품 수: 200


NameError: name 'category_name' is not defined

In [12]:
-category 별로 클릭해 들어가서 상품들 스크롤하기.
-출력: 카테고리ID, 카테고리 이름, 상품이름
-mysql 카테고리 테이블에 저장
기존 db에 들어가서 JOIN 하기



from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time
import pymysql

# mysql 접속정보
conn = pymysql.connect(
    host="127.0.0.1",
    user="root",
    password="Hayeon123!",
    db="wconcept_crawling02",
    charset="utf8mb4"
)

cursor = conn.cursor()

# 가상 브라우저 세팅정보
service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수:", len(items))

def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0


 #카테고리 & 상품 정보
elements = driver.find_element(By.CSS_SELECTOR,"div.swiper-slide")

for element in elements :
    category_id = element.get_attribute("data-value")
    category_name = element.find_element(By.CSS_SELECTOR,"sapn.mnbrtm-inner").get.text.strip()
print(element)


# #mysql 테이블 저장 -categories
#     cursor.execute("""
#         INSERT INTO categories (brand_id, category_name, category_id) VALUES (%s, %s, %s) 
#     """, (brand_id, category_name, category_id))

찾은 상품 수: 200


TypeError: 'WebElement' object is not iterable